# 2.5 — MyoReflex: spinal-feedback walking

This notebook runs the **Song-Geyer** reflex controller on `myoLegWalk-v0`. Muscle excitations come from spinal feedback (length, force, and ground load), not from a learned policy.

**What you'll learn:**
- How to load the published 46-parameter gain vector
- How `MyoLegReflex` wraps a CPU Gymnasium env and steps the reflex
- How to render an offscreen rollout

**Prerequisites:** Completed notebook 1.1 · run this notebook from `tutorials/` or `tutorials/files/2.5/`

> Citation: Song & Geyer, *J Physiol* 2015. The wrapper maps the 80 MyoLeg muscles onto the original 11-per-leg groups. With the published gains the current MyoLeg+torso model takes several forward steps; it is not a fully stabilized 3D gait.


In [ ]:
import sys
from pathlib import Path

import numpy as np

from myosuite.utils.video_io import show_video, write_video

_here = Path.cwd()
if (_here / "reflex_ctr_interface.py").exists():
    sys.path.insert(0, str(_here))
elif (_here / "files/2.5" / "reflex_ctr_interface.py").exists():
    sys.path.insert(0, str(_here / "files/2.5"))
    _here = _here / "files/2.5"
else:
    raise FileNotFoundError(
        "Run from tutorials/ or tutorials/files/2.5/ so reflex_ctr_interface.py is importable."
    )

import reflex_ctr_interface


Load `baseline_params.txt`, reset the walker, and roll out. `normalize_act=False` so the reflex can write raw `[0, 1]` excitations. Rendering uses Gymnasium `rgb_array` (free camera).

In [ ]:
sim_time = 2  # seconds; raise to 5 for a longer clip
dt = 0.01
steps = int(sim_time / dt)
frames = []

params = np.loadtxt(_here / "baseline_params.txt")

Myo_env = reflex_ctr_interface.MyoLegReflex(sim_time=sim_time, seed=0)
Myo_env.reset()
Myo_env.set_control_params(params)

for timestep in range(steps):
    frame = Myo_env.env.render()
    if frame is not None:
        frames.append(frame)
    _, is_done, _, _ = Myo_env.run_reflex_step()
    if is_done:
        print(f"Stopped at step {timestep} ({timestep * dt:.2f} s)")
        break

Myo_env.env.close()

out_mp4 = _here / "MyoReflex_output.mp4"
if frames:
    write_video(out_mp4, np.asarray(frames), fps=50, outputdict={"-pix_fmt": "yuv420p"})
    print(f"Wrote {out_mp4} ({len(frames)} frames)")


In [ ]:
show_video(str(out_mp4))
